In [ ]:
from src.preprocess.parquet_preprocessor import ParquetPreprocessor

ParquetPreprocessor.csv_to_parquet("data/raw/active_alarms_prod.csv", "data/raw/active_alarms_prod.parquet")

In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/active_alarms_prod.parquet")
graph_repo = AlarmGraphRepository(os.getenv("ACTIVE_DB_PATH"))

In [2]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationActive

SimpleTimeCorrelationActive.train(lazy_frame, graph_repo)

Processando nós: 100%|██████████| 866/866 [00:13<00:00, 63.69nó/s] 


In [4]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

Calculando WCC: 100%|██████████| 866/866 [00:03<00:00, 288.51nó/s]


alert_id,node_id,alert_type,start_time,end_time,incident
str,str,str,date,date,i32
"""305da354-2edf-444e-a2b7-4edb06…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_QLE…",2025-12-05,2025-12-05,0
"""8a6acea4-c8db-46f8-8c6a-0ca91a…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_CUR""",2025-12-05,2025-12-05,0
"""514ed2e4-e107-4d2b-82ba-a232c6…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_AVG""",2025-12-05,2025-12-05,0
"""4ef9274c-e078-468f-8b3f-dc44d3…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_PHYSICAL_DISK_ALLOCATI…",2025-11-11,2025-12-05,0
"""88b82f4c-e0a0-4097-844a-84d5b3…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_AVG""",2025-12-05,2025-12-05,0
…,…,…,…,…,…
"""a9518d63-8bd3-40eb-ab46-710461…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_KBYTES_P…",2025-12-05,2025-12-05,0
"""d046d918-6938-4705-9caf-672fbd…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IO_PER_S…",2025-12-05,2025-12-05,0
"""08d790f7-1153-456b-983f-80d267…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_KBYTES_P…",2025-12-05,2025-12-05,0


In [2]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_active")
results_repo.save(summary)
results_repo.load()

✅ Nova versão salva com sucesso em: data/results/simple_time_corr_active_20260519_180956.csv
📖 Carregando a versão mais recente encontrada: simple_time_corr_active_20260519_180956.csv


Node ID,Total de Alarmes,Total de Correlações,Total de Incidentes,Média de Alarmes por Incidente,Densidade
str,i64,i64,i64,f64,f64
"""c60cc1af-aa30-491a-a3ef-55c6f8…",399,417,130,3.069231,0.005252
"""f3ba8721-2bb0-46b3-b4dd-c9255b…",237,12358,18,13.166667,0.441894
"""4a278aa6-9cfe-4d85-ae05-9843e5…",60,4,4,15.0,0.00226
"""4c43b243-0774-4296-8f18-85c699…",57,4,4,14.25,0.002506
"""3a15506c-b858-4cd5-8bba-8fafdc…",60,4,4,15.0,0.00226
…,…,…,…,…,…
"""068f4329-2d28-4e6f-b4bf-7249f4…",1,0,1,1.0,0.0
"""76570354-6bff-4143-91e0-7e6c9b…",1,0,1,1.0,0.0
"""c9a3f847-3474-4352-bd42-c6fbe7…",1,0,1,1.0,0.0
